In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import LabelEncoder

horse_colic = fetch_ucirepo(id=47)

X = horse_colic.data.features.copy()
y = horse_colic.data.targets.copy()

df = pd.concat([X, y], axis=1)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace('-', '_', regex=False)
    .str.replace(' ', '_', regex=False)
)

target_col = y.columns[0]
target_col = (
    target_col.strip()
    .lower()
    .replace('-', '_')
    .replace(' ', '_')
)

df_original = df.copy()

print('Размер датасета:', df.shape)
print(df.head())

Размер датасета: (368, 28)
   surgery  age  hospital_number  rectal_temperature  pulse  respiratory_rate  \
0      2.0    1           530101                38.5   66.0              28.0   
1      1.0    1           534817                39.2   88.0              20.0   
2      2.0    1           530334                38.3   40.0              24.0   
3      1.0    9          5290409                39.1  164.0              84.0   
4      2.0    1           530255                37.3  104.0              35.0   

   temperature_of_extremities  peripheral_pulse  mucous_membranes  \
0                         3.0               3.0               NaN   
1                         NaN               NaN               4.0   
2                         1.0               1.0               3.0   
3                         4.0               1.0               6.0   
4                         NaN               NaN               6.0   

   capillary_refill_time  ...  packed_cell_volume  total_protein  \
0  

In [2]:
label_encoders = {}

for column in df.columns:
    if df[column].dtype == 'object':
        label_encoder = LabelEncoder()
        df[column] = label_encoder.fit_transform(df[column].astype(str))
        label_encoders[column] = label_encoder

print("\nСтатистика данных:")
print(df.describe())

print("\nПроверка пропусков:")
print(df.isnull().sum())

print("\nРаспределение целевой переменной class:")
print(df[target_col].value_counts())

numeric_df = df.select_dtypes(include='number')
correlation_matrix_spearman = numeric_df.corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_spearman, annot=True, cmap='coolwarm')
plt.title("Корреляция Спирмена между признаками")
plt.show()

NameError: name 'feature_importances_table' is not defined

In [ ]:
from sklearn.ensemble import RandomForestClassifier

df_encoded = df.copy()
label_encoders = {}

for column in df_encoded.columns:
    if df_encoded[column].dtype == 'object':
        le = LabelEncoder()
        df_encoded[column] = le.fit_transform(df_encoded[column].astype(str))
        label_encoders[column] = le

X = df_encoded.drop([target_col], axis=1)
y = df_encoded[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

feature_importances = model.feature_importances_
sorted_idx = feature_importances.argsort()

plt.figure(figsize=(9, 8))
plt.barh(X.columns[sorted_idx], feature_importances[sorted_idx])
plt.xlabel('Feature Importance')
plt.title('Важность признаков для определения целевой переменной')
plt.tight_layout()
plt.show()

feature_importances_table = pd.DataFrame({
    'Признак': X.columns,
    'Важность': feature_importances
}).sort_values(by='Важность', ascending=False)

print(feature_importances_table)

selector = SelectFromModel(model, prefit=True, threshold='median')
selected_features = X.columns[selector.get_support()]

print("\nОтобранные признаки:")
print(list(selected_features))

selected_columns = list(selected_features) + [target_col]
df_reduced = df_original[selected_columns]

df_reduced.to_csv(
    "horse_colic_reduced.csv",
    index=False
)

print("Сокращённый датасет сохранён")
print(df_reduced.shape)
print(df_reduced.head())